# 4주차 팀과제 — 14조 · 미션 D (방과후 청소년 공간)

**의사결정자**: 청소년정책 담당자 · **결정할 것**: 방과후 청소년이 많이 머무는 자치구에 공간 신설
**범위**: 15~18시 · 10대 · **절대**: 10대 합계 · **비중**: 같은 시간대 전 연령대 대비

## STEP① 질문 정의 — 가설 문장 (초안담당자: 정승엽 / 15점)

**가설**: 오후 방과후 시간대(15~18시)의 10대 생활인구는 거주지가 아니라 학원·학교 등 사교육/교육시설이 밀집한 자치구에 몰릴 것이다. 따라서 10대 절대 합계 상위 3개 자치구에는 서울의 대표적 학원가로 알려진 강남구·양천구·노원구가 다수 포함될 것이다. 다만 비중(같은 시간대 전 연령대 대비 10대 비율) 상위 자치구는, 강남구·송파구처럼 업무·상업 유동인구까지 많아 비중이 희석되는 번화한 학원가보다, 학원가이면서 동시에 전통적으로 주거 비중이 높은(이른바 '베드타운') 노원구처럼 다른 연령대 유동인구가 상대적으로 적은 자치구에서 더 두드러질 것이다.

> 위 자치구 선정 근거(학원 수·주간인구지수 등 외부 자료)는 `00_가설_질문정의.md` 3-1절 참고.

**반대 결과가 나와도 받아들일 수 있는가(1줄 필수)**: 받아들인다 — 절대·비중 상위 자치구가 위 학원가 목록(강남·양천·노원)과 무관하게 나오면 '학원가 밀집 가설'을 기각하고 공원·대형 상업시설·환승거점·산업단지 통근 인구 등 비학업적 체류 요인으로 설명을 바꾸며, 이 데이터가 2026-04-15 단 하루·교육용으로 오류가 삽입된 가공 데이터라는 한계도 결론에 함께 명시한다.


In [5]:
# 0. 환경 준비
!pip install -q koreanize-matplotlib
import koreanize_matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


## STEP② 데이터 이해 (초안담당자: 정승엽 / 10점)

`docs/수집_기록표.md`의 수집 기록표·컬럼 5개 측정 척도를 참고하세요.


In [13]:
import os

# Colab(파일을 노트북과 같은 위치에 업로드)과 로컬(data/ 폴더에 원본 보존) 경로가 달라서 둘 다 시도
_candidates = [
    '서울시_생활인구_202604_dl0830.csv',
    '../data/서울시_생활인구_202604_dl0830.csv',
    'data/서울시_생활인구_202604_dl0830.csv',
]
_path = next(p for p in _candidates if os.path.exists(p))

df = pd.read_csv(_path, encoding='cp949')
df.columns = [c.strip() for c in df.columns]
print('불러온 경로:', _path)
print('행, 열:', df.shape)
print('=' * 50)
print('자료형 파악: ' )
df.info()
print('=' * 50)

df.head(10)


불러온 경로: ../data/서울시_생활인구_202604_dl0830.csv
행, 열: (3860, 5)
자료형 파악: 
<class 'pandas.DataFrame'>
RangeIndex: 3860 entries, 0 to 3859
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   기준일자    3860 non-null   str  
 1   자치구     3860 non-null   str  
 2   시간대     3860 non-null   int64
 3   연령대     3860 non-null   str  
 4   생활인구수   3765 non-null   str  
dtypes: int64(1), str(4)
memory usage: 150.9 KB


,기준일자,자치구,시간대,연령대,생활인구수
0,2026-04-15,강북구,7,30대,3287
1,2026-04-15,구로구,15,10대,4515
2,2026-04-15,동대문구,14,40대,2546
3,2026-04-15,광진구,2,60대이상,1150
4,2026-04-15,서초구,13,20대,2456
5,2026-04-15,서대문구,5,20대,462
6,2026-04-15,서대문구,2,60대이상,462
7,2026-04-15,용산구,17,40대,3547
8,2026-04-15,강남구,15,30대,9016
9,2026-04-15,서초구,6,30대,1453


In [ ]:
# 컬럼별로 값을 살펴보면서 자료구조 파악 (측정 척도 판단용)

print('[기준일자] 고유값:', df['기준일자'].unique())

print('\n[자치구] 값 개수:', df['자치구'].nunique())
print(sorted(df['자치구'].unique()))

print('\n[시간대] 범위:', df['시간대'].min(), '~', df['시간대'].max(), '| dtype:', df['시간대'].dtype)

print('\n[연령대] 값 목록:', df['연령대'].unique())

print('\n[생활인구수] dtype:', df['생활인구수'].dtype)
_comma_example = df.loc[df['생활인구수'].astype(str).str.contains(','), '생활인구수'].head(2).tolist()
_has_dash = (df['생활인구수'] == '-').any()
print('콤마 포함 예시:', _comma_example, "| '-' 표시 존재 여부:", _has_dash)
# 생활인구수 컬럼은 콤마 제거 후 int로 변환 필요
# 이 정보는 '6대 품질 진단'과 '이슈별 상세 근거'도출 과정을 프로젝트 세팅과정에서 진행한 배경지식을 토대로 진행하였다.


[기준일자] 고유값: <StringArray>
['2026-04-15']
Length: 1, dtype: str

[자치구] 값 개수: 25
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '동작구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구']

[시간대] 범위: 0 ~ 23 | dtype: int64

[연령대] 값 목록: <StringArray>
['30대', '10대', '40대', '60대이상', '20대', '50대']
Length: 6, dtype: str

[생활인구수] dtype: str
콤마 포함 예시: ['3,909', '3,552'] | '-' 표시 존재 여부: True


## STEP③ 탐색 — 첫인상 확인 + 이상 징후 3개 이상 + 정제 전 1차 집계 (초안담당자: 정승엽 / 15점)

AI에게 시킬 프롬프트 예시: "이 CSV의 첫 10줄, 행·열 개수, 각 컬럼의 자료형과 결측 개수를 보여줘. 이상해 보이는 값도 함께 지적해줘."


In [8]:
# 자료형 · 결측 개수
print(df.dtypes)
print()
print(df.isnull().sum())


기준일자       str
자치구        str
시간대      int64
연령대        str
생활인구수      str
dtype: object

기준일자      0
자치구       0
시간대       0
연령대       0
생활인구수    95
dtype: int64


In [ ]:
# 이상 징후 기록 (최소 3개 — docs/data_description.md 참고)

# 1) 생활인구수 결측(빈칸) — 어떤 연령대에 몰려 있는지
null_mask = df['생활인구수'].isnull()
null_by_age = df.loc[null_mask, '연령대'].value_counts()
print(f'1) 생활인구수 결측 {null_mask.sum()}건 — 연령대별 분포:')
print(null_by_age)

# 2) 완전 중복행(모든 컬럼 동일) — 몇 번씩 중복되는지
dup_mask = df.duplicated(keep=False)
dup_rows = dup_mask.sum()
dup_group_counts = df[dup_mask].groupby(list(df.columns)).size()
print(f'\n2) 완전 중복행 {dup_rows}행 (중복 쌍 {dup_rows // 2}개) — 그룹별 반복 횟수 분포:', dup_group_counts.value_counts().to_dict())

# 3) 생활인구수 음수 — -9990 같은 특이값 구분
numeric_try = pd.to_numeric(df['생활인구수'].astype(str).str.replace(',', ''), errors='coerce')
neg_mask = numeric_try < 0
special_mask = numeric_try == -9990
print(f'\n3) 생활인구수 음수 {neg_mask.sum()}건 (그중 -9990 특이값 {special_mask.sum()}건)')

# 4) 생활인구수 '-' 문자 표시
dash_mask = df['생활인구수'] == '-'
dash_by_age = df.loc[dash_mask, '연령대'].value_counts()
print(f"\n4) 생활인구수 '-' 표시 {dash_mask.sum()}건 — 연령대별 분포:")
print(dash_by_age)

# 5) 생활인구수 비정상 고값 — 어느 자치구·시간대에 몰려 있는지
high_mask = numeric_try > 100000
print(f'\n5) 생활인구수 10만 초과 {high_mask.sum()}건')
if high_mask.sum() > 0:
    print(df.loc[high_mask, ['자치구', '시간대', '연령대', '생활인구수']])

# 6) 컬럼명 공백 — 원본 헤더 기준(지금 df는 로드 시 이미 strip 처리됨)
raw_cols = pd.read_csv(_path, encoding='cp949', nrows=0).columns.tolist()
print(f'\n6) 원본 컬럼명(공백 포함 여부 확인): {raw_cols}')

# 7) 생활인구수 콤마(,) 포함 여부 혼재
comma_mask = df['생활인구수'].astype(str).str.contains(',')
no_comma_high = ((~comma_mask) & numeric_try.ge(1000)).sum()
print(f'\n7) 콤마 포함 {comma_mask.sum()}건 / 콤마 없이 1,000 이상인 값 {no_comma_high}건 — 표기 방식이 혼재')


In [ ]:
# 정제 전 1차 집계 — 우리 미션 범위(15~18시 · 10대)만 먼저 필터링해서 행 수를 확인
# "15~18시"는 15·16·17시 3개 시간대를 뜻함(18시는 18:00~19:00 버킷이라 범위 밖) — 00_가설_질문정의.md 참고
before = df[(df['시간대'].between(15, 17)) & (df['연령대'].str.strip() == '10대')]
print('정제 전 미션 범위 행 수:', len(before))
before.head()


## STEP④ 정제 — 정제 로그 5줄 이상 (처리·건수·근거·영향) (초안담당자: 정승엽 / 25점, 가장 큰 배점)

`logs/정제로그.md` 초안을 참고해 팀이 실제로 결정한 내용으로 코드를 작성하세요.
**몇 건을 지웠는지가 아니라 '왜' 지웠는지/안 지웠는지가 점수입니다.**


### 정제 원칙과 순서

- **범위**: "15~18시" = **15·16·17시 3개 시간대**(18시 버킷 제외, STEP③ 합의). 절대(분자)는 **10대**, 비중(분모)은 같은 시간대 **전 연령대**를 쓴다.
- **원칙**: 이슈를 전부 고치지 않는다. 먼저 *범위 안에 몇 건 있는지* 세고, 영향이 있을 때만 고친다. 손대지 않은 것도 이유를 남긴다.
- **순서**
  1. **사전 진단(데이터는 바꾸지 않음)** — 미션 범위 기준으로 행 수와 이슈별 건수를 **원본 전체 → 범위(전 연령대·10대) → 중복 제거 후**로 센다. 범위 안에 영향이 있는지 보고 처리 여부를 정하기 위해서다.
  2. **정제** — 열별 형식 정리(공백·숫자화) → 열과 무관한 정제(중복 행) → 열별 값 오류(음수·결측) → 손대지 않는 것 확인. 정제는 전체 표에 하고, 영향은 범위 기준으로 추적한다.
  3. **분석용 필터링은 정제가 끝난 뒤**(STEP⑤)에 한다.
  4. 마지막에 **단계별 추적표**와 정제 로그로 "무엇이 몇 건에서 몇 건이 됐는지" 한눈에 확인한다.
- 숫자화를 중복 제거보다 먼저 하는 이유: `1,234`와 `1234`처럼 표기만 다른 중복도 같은 값으로 잡힌다. 중복 제거를 결측 보정보다 먼저 하는 이유: 중복 행이 평균을 왜곡하지 않게 하기 위해서다.

In [ ]:
# ── 정제 준비 ───────────────────────────────────────────────
H_START, H_END = 15, 17      # '15~18시' = 15·16·17시 3개 시간대 (STEP③ 합의)
TARGET_AGE = '10대'
N_GU = df['자치구'].nunique()
N_HOURS = H_END - H_START + 1
N_AGES = df['연령대'].str.strip().nunique()

def in_scope(d):             # 시간대 범위 마스크 (연령 제한 없음)
    return d['시간대'].between(H_START, H_END)

def mission(d):              # 미션 범위: 15~17시 · 10대 (절대의 분자)
    return d[in_scope(d) & (d['연령대'].str.strip() == TARGET_AGE)]

clean = df.copy()            # 원본 df는 건드리지 않고, 이후 정제는 전부 clean에만 적용
log = []                     # 정제 로그(처리·건수·근거·영향) — 마지막 셀에서 표로 출력

## 0. 사전 진단 — 미션 범위 기준으로 행 수·이슈 건수 세기 (데이터는 바꾸지 않음)

절대는 **10대만**, 비중은 분모가 **같은 시간대 전 연령대**라서 연령을 제한하지 않은 범위도 함께 본다. 정제 전에 행 수가 기대값(구×시간대×연령대)과 맞는지, 이슈가 범위 안에 몇 건 있는지 확인한다. `clean`은 아직 건드리지 않는다.

In [ ]:
# 사전 진단용 범위 보기 (원본 df 기준, clean은 아직 건드리지 않음)
scope_teen = df[in_scope(df) & (df['연령대'].str.strip() == TARGET_AGE)]   # 15~17시 · 10대    → 절대(분자)
scope_all  = df[in_scope(df)]                                              # 15~17시 · 전 연령대 → 비중(분모), 연령 제한 없음

dedup = df.drop_duplicates()                                               # 중복 제거 후 행 수를 비교하기 위한 사본(정제 본작업은 아래에서 따로)
def _teen_rows(d): return int((in_scope(d) & (d['연령대'].str.strip() == TARGET_AGE)).sum())

funnel = pd.DataFrame({
    '원본 행 수':        [len(df),    len(scope_all),            len(scope_teen)],
    '중복 제거 후 행 수': [len(dedup), int(in_scope(dedup).sum()), _teen_rows(dedup)],
    '기대값(구×시간대×연령대)': [N_GU * 24 * N_AGES, N_GU * N_HOURS * N_AGES, N_GU * N_HOURS],
}, index=['전체', f'범위 {H_START}~{H_END}시 · 전 연령대 (비중 분모용)', f'범위 {H_START}~{H_END}시 · {TARGET_AGE} (절대 분자용)'])
display(funnel)
assert len(scope_teen) > 0 and len(scope_all) > 0, '필터 결과가 0행 — 코드가 아니라 조건(시간대·연령대 값)을 의심'

In [ ]:
# 이슈별 건수 추적 도구 — "전체에서 몇 건 → 범위 안에서 몇 건 → 중복 제거 후 몇 건"을 같은 기준으로 센다
def make_flags(d):                       # d: 원본 표기(문자열) 그대로의 표
    s = d['생활인구수']
    num = pd.to_numeric(s.str.replace(',', '', regex=False).str.strip(), errors='coerce')
    is_dash, is_blank = s.str.strip() == '-', s.isnull()
    return pd.DataFrame({
        '완전 중복 행(2회째)': d.duplicated(),
        '결측 합계':           is_dash | is_blank,
        '  결측: 빈칸':        is_blank,
        "  결측: '-'":          is_dash,
        '음수(부호 오류 추정)': (num < 0) & (num != -9990),
        '특수코드 -9990':      num == -9990,
        '고값(>10만)':         num > 100000,
        '콤마 포함(형식)':     s.str.contains(',', regex=False),
    }, index=d.index)

def count_table(d):                      # 이슈 × (전체 / 범위·전 연령 / 범위·10대)
    f = make_flags(d)
    sc = in_scope(d)
    tn = sc & (d['연령대'].str.strip() == TARGET_AGE)
    t = pd.DataFrame({'전체': f.sum(), '범위·전 연령': f[sc].sum(), '범위·10대': f[tn].sum()})
    ref = pd.DataFrame({'전체': [len(d)], '범위·전 연령': [int(sc.sum())], '범위·10대': [int(tn.sum())]}, index=['(참고) 행 수'])
    return pd.concat([ref, t]).astype(int)

t1 = count_table(df)                     # ① 원본
t2 = count_table(dedup)                  # ② 중복 제거 후

def flow(issue):                         # 정제 로그용 한 줄: 원본 전체 → 범위 → 중복 제거 후 범위
    a, b = t1.loc[issue], t2.loc[issue]
    return (f"원본 전체 {a['전체']} → 범위 {a['범위·전 연령']}(10대 {a['범위·10대']})"
            f" → 중복 제거 후 전체 {b['전체']}, 범위 {b['범위·전 연령']}(10대 {b['범위·10대']})")

print('① 원본 기준 이슈별 건수 (정제 전)')
display(t1)

## A. 열별 정제 — 형식 정리

In [ ]:
# ① 컬럼명·문자열 값 앞뒤 공백 (일관성) — 안 고치면 groupby·필터에서 KeyError / 값 불일치
raw_cols = pd.read_csv(_path, encoding='cp949', nrows=0).columns.tolist()
n_bad_cols = sum(c != c.strip() for c in raw_cols)

n_strip = 0
for c in ['기준일자', '자치구', '연령대']:
    n_strip += int((clean[c] != clean[c].str.strip()).sum())
    clean[c] = clean[c].str.strip()

assert all(c == c.strip() for c in clean.columns), '컬럼명에 공백이 남아 있음'
print('원본 컬럼명:', raw_cols)
print('공백 있던 컬럼명:', n_bad_cols, '개 → 로드 시 strip 처리 완료')
print('값(자치구·연령대·기준일자)에서 공백이 제거된 셀:', n_strip, '개')
print('자치구 고유값', clean['자치구'].nunique(), '개 | 연령대 고유값', clean['연령대'].nunique(), '개')

log.append(dict(처리='컬럼명 앞뒤 공백 제거(strip)', 건수=f'컬럼명 {n_bad_cols}개 / 값 {n_strip}셀',
    근거='원본 헤더에 공백이 있어 groupby·필터 시 KeyError 유발 (일관성)',
    영향='채택 — 이후 모든 코드가 깨지지 않으려면 가장 먼저 처리. 값 쪽 공백은 없어 결과 영향 없음'))

In [ ]:
# ② 생활인구수: 콤마 제거 → 숫자형 (일관성·유효성) — 문자열(object)이면 합계·평균이 안 됨
raw = clean['생활인구수']
n_comma = int(raw.str.contains(',', regex=False).sum())
n_dash  = int((raw.str.strip() == '-').sum())
n_blank = int(raw.isnull().sum())

clean['생활인구수'] = pd.to_numeric(raw.str.replace(',', '', regex=False).str.strip(), errors='coerce')
n_nan = int(clean['생활인구수'].isnull().sum())

print(f"콤마 포함 {n_comma}건 → 제거 후 숫자 변환 | 변환 후 dtype: {clean['생활인구수'].dtype}")
print(f"변환 후 NaN {n_nan}건 = '-' {n_dash}건 + 빈칸 {n_blank}건")
# errors='coerce'는 이상한 문자열도 조용히 NaN으로 바꾼다 → 예상한 두 종류(‘-’·빈칸)만 NaN이 됐는지 확인
assert n_nan == n_dash + n_blank, "'-'·빈칸 외의 값이 조용히 NaN이 됨 — 원인 확인 필요"

log.append(dict(처리='생활인구수 콤마 제거 후 숫자형 변환', 건수='콤마 ' + flow('콤마 포함(형식)'),
    근거='천단위 콤마가 있는 값과 없는 값이 섞여 문자열 타입이라 합계·평균 계산 불가 (일관성)',
    영향=f"채택 — 안 하면 집계 자체가 안 됨. 변환 후 NaN {n_nan}건은 전부 '-'·빈칸이라 임의 손실 없음 (assert 확인)"))

## B. 열과 무관한 정제 — 완전 중복 행 (모든 컬럼이 같은 행)

In [ ]:
# ⑥ 완전 중복 행 제거 (유일성) — 합계가 부풀려지는 것을 막기 위해 필수
KEYS = ['기준일자', '자치구', '시간대', '연령대']
dup = clean.duplicated(keep='first')
n_dup = int(dup.sum())
n_dup_scope = int((dup & in_scope(clean)).sum())
n_dup_teen_scope = int((dup & in_scope(clean) & (clean['연령대'] == TARGET_AGE)).sum())

# 같은 키(구·시간대·연령대)인데 값만 다른 '충돌 행'이 있는지 — 있으면 단순 제거가 아니라 판단이 필요
n_key_dup = int(clean.duplicated(KEYS, keep='first').sum())
assert n_key_dup == n_dup, '키는 같지만 값이 다른 행이 있음 — 중복 제거만으로는 부족'

n_before = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
print(f'전체: {n_before}행 → {len(clean)}행 (제거 {n_dup}행)')
print(f'범위(15~17시) 안 중복 {n_dup_scope}행, 그중 10대 {n_dup_teen_scope}행 → 합계에 직접 영향')
print('미션 범위(15~17시·10대) 필터 후 행 수:', len(mission(clean)), '(기대:', N_GU * N_HOURS, ')')
assert len(mission(clean)) == N_GU * N_HOURS, '자치구×시간대 조합 수와 다름 — 중복/누락 점검'
assert len(clean) == len(dedup), '위에서 비교용으로 만든 dedup과 행 수가 다름'

log.append(dict(처리='완전 중복 행 제거(drop_duplicates)',
    건수=f'원본 전체 {n_dup}행 → 범위 {n_dup_scope}행(10대 {n_dup_teen_scope}행)',
    근거='기준일자·자치구·시간대·연령대·값이 모두 같은 행이 2회씩 등장, 같은 키에 다른 값인 충돌 행은 0건 (적재 중복 입력으로 추정)',
    영향=f'채택 — 안 하면 10대 합계가 {n_dup_teen_scope}행만큼 부풀려짐. 제거 후 미션 범위 {N_GU}구×{N_HOURS}시간대={N_GU * N_HOURS}행(10대), 전 연령대 {N_GU * N_HOURS * N_AGES}행으로 기대값과 일치'))

## C. 열별 값 오류 — 생활인구수 음수

In [ ]:
# ⑤ 음수 (유효성) — 인구수는 음수일 수 없다. 범위 안에 있는지부터 센다
v = clean['생활인구수']
neg = v < 0
is_code = v == -9990                 # 특수 오류코드로 추정
is_sign = neg & ~is_code             # 부호만 잘못 붙은 것으로 추정
neg_scope = neg & in_scope(clean)

print(f'음수 {int(neg.sum())}건(중복 제거 후) = 특수코드(-9990) {int(is_code.sum())}건 + 그 외 {int(is_sign.sum())}건')
print(f'범위(15~17시) 안 음수: {int(neg_scope.sum())}건 | 그중 10대: {int((neg_scope & (clean["연령대"] == TARGET_AGE)).sum())}건 → 10대 합계(절대)에는 영향 없음, 비중 분모에만 영향')
print('10대 음수(전체 시간대):', clean.loc[neg & (clean['연령대'] == TARGET_AGE), ['자치구', '시간대', '생활인구수']].values.tolist(), '→ 범위 밖')

# 부호 복원이 타당한가: |값|이 같은 자치구·같은 연령대의 다른 시간대 정상값 범위 안에 들어오는가
ok_mask = v >= 0
rows = []
for i, r in clean[neg_scope].iterrows():
    peer = clean.loc[ok_mask & (clean['자치구'] == r['자치구']) & (clean['연령대'] == r['연령대']), '생활인구수']
    rows.append(dict(자치구=r['자치구'], 시간대=r['시간대'], 연령대=r['연령대'], 원값=r['생활인구수'], 절대값=abs(r['생활인구수']),
                     같은구연령_정상값_최소=peer.min(), 같은구연령_정상값_최대=peer.max(),
                     범위안=bool(peer.min() <= abs(r['생활인구수']) <= peer.max())))
display(pd.DataFrame(rows))

In [ ]:
# 적용: 특수코드(-9990)는 결측 처리, 나머지 음수는 부호만 복원(abs)
n_code, n_sign = int(is_code.sum()), int(is_sign.sum())
n_neg_scope = int(neg_scope.sum())
clean.loc[is_code, '생활인구수'] = np.nan
clean.loc[is_sign, '생활인구수'] = clean.loc[is_sign, '생활인구수'].abs()
assert (clean['생활인구수'].dropna() >= 0).all(), '음수가 남아 있음'
print('적용 후 음수:', int((clean['생활인구수'] < 0).sum()), '건 | 새로 생긴 결측(-9990):', n_code, '건')

log.append(dict(처리='음수 처리: -9990은 결측, 나머지는 abs()로 부호 복원',
    건수='부호 오류 ' + flow('음수(부호 오류 추정)') + ' / -9990 ' + flow('특수코드 -9990'),
    근거='-9990은 정상 인구수 범위 밖의 특수 오류코드로 추정. 그 외 음수의 절대값(417~7,804)은 정상 인구수 범위와 비슷하고, 범위 안 음수는 같은 구·연령대 다른 시간대 값 범위 안이라 부호만 잘못 붙은 것으로 판단',
    영향=f'채택 — 범위 안 {n_neg_scope}건은 모두 10대가 아니라 비중 분모에만 영향. 10대 음수 1건(성북구 13시)과 -9990(강남구 12시)은 범위 밖이라 결과 영향 없음(규칙 일관성을 위해 같이 적용)'))

## D. 열별 값 오류 — 생활인구수 결측(`'-'`·빈칸)

`'-'`와 빈칸은 표기만 다른 같은 종류의 결측이라 함께 다룬다. **절대(10대 합계)는 분자, 비중은 분자÷분모(전 연령대 합계)** 이므로, 결측이 분자에 있는지 분모에 있는지를 나눠서 본다.

In [ ]:
# ③·④ 결측 진단 — 범위 안/밖, 10대/그 외로 나눠서 센다
miss = clean['생활인구수'].isnull()
scope = in_scope(clean)
teen = clean['연령대'] == TARGET_AGE

print(f'결측 전체 {int(miss.sum())}건(중복 제거 후, -9990 결측 포함)')
print(f'  범위 밖 {int((miss & ~scope).sum())}건 → 결과에 영향 없음, 손대지 않음')
print(f'  범위 안 {int((miss & scope).sum())}건 (전 연령대) → 이 중 10대(분자) {int((miss & scope & teen).sum())}건, 그 외(분모) {int((miss & scope & ~teen).sum())}건')
print(f"  10대 결측 {int((miss & teen).sum())}건 중 범위 안은 {int((miss & teen & scope).sum())}건, 범위 밖 {int((miss & teen & ~scope).sum())}건")

# 원본 표기('-' / 빈칸)별로 연령대 분포 확인 (범위 안만)
raw_u = df.drop_duplicates()
kind = pd.Series(np.where(raw_u['생활인구수'].astype(str).str.strip() == '-', "'-'",
                 np.where(raw_u['생활인구수'].isnull(), '빈칸', '값있음')), index=raw_u.index, name='표기')
print("\n[범위 안 · 표기별 · 연령대별 건수] ('값있음'은 결측 아님)")
display(pd.crosstab(raw_u.loc[in_scope(raw_u), '연령대'], kind[in_scope(raw_u)]))

print('분모(전 연령대 합계)에 결측이 있는 자치구 수:', clean.loc[miss & scope & ~teen, '자치구'].nunique(), '/', N_GU)

# 분자(10대) 결측이 범위 안에 있다면 어느 구인지 먼저 확인 — 지금은 0건이어야 정상
teen_miss_scope = clean.loc[miss & scope & teen, ['자치구', '시간대', '연령대']]
display(teen_miss_scope)
assert len(teen_miss_scope) == 0, '범위 안 10대 결측 발생 — 어느 구인지 확인하고 보정 방식을 정해야 함 (범위/시간대 정의가 바뀌었는지 점검)'

In [ ]:
# 분모 결측을 보정할지 말지 — 두 방식의 결과를 비교한 뒤 팀이 '보정함'으로 확정(2026-10-09)
#   'skip'   : 보정하지 않음. 결측 셀은 합계에서 빠진다 → 그 구의 분모(전 연령대 합계)가 작아져 비중이 부풀려질 수 있음
#   'impute' : 같은 자치구·같은 연령대의 범위 내(15~17시) 다른 시간대 평균으로 대체 (범위 안에서만, 범위 밖은 그대로)
DENOM_MISSING = 'impute'

def summarize(d):
    m = d[in_scope(d)]
    teen_sum = m[m['연령대'] == TARGET_AGE].groupby('자치구')['생활인구수'].sum()
    total = m.groupby('자치구')['생활인구수'].sum()
    return teen_sum, teen_sum / total * 100

def impute_in_scope(d):
    d = d.copy()
    s = in_scope(d)
    peer_mean = d[s].groupby(['자치구', '연령대'])['생활인구수'].transform('mean')
    d.loc[s, '생활인구수'] = d.loc[s, '생활인구수'].fillna(peer_mean)
    return d

abs_skip, share_skip = summarize(clean)
imputed = impute_in_scope(clean)
abs_imp, share_imp = summarize(imputed)

print('[절대] 10대 합계가 보정 전후 같은가:', bool(np.allclose(abs_skip, abs_imp)), '→ 절대 순위는 결측 처리와 무관')
print('[절대] 상위5:', abs_imp.sort_values(ascending=False).head(5).round(0).astype(int).to_dict())

def top5_text(sr):
    return [f"{gu} {val:.2f}%" for gu, val in sr.sort_values(ascending=False).head(5).items()]

cmp5 = pd.DataFrame({'비중 상위5 (보정 안 함)': top5_text(share_skip),
                     '비중 상위5 (보정함)': top5_text(share_imp)}, index=range(1, 6))
display(cmp5)
rank_changed = int((share_skip.rank(ascending=False) != share_imp.rank(ascending=False)).sum())
top5_same = set(share_skip.nlargest(5).index) == set(share_imp.nlargest(5).index)
print(f'[비중] 순위가 달라진 자치구 {rank_changed}/{N_GU}개 | 상위5 구성 동일 여부: {top5_same}')

In [ ]:
# 결정 적용: 범위 안 분모 결측(빈칸 + '-')을 같은 구·연령대 평균으로 채움
n_filled = int((miss & scope).sum())
n_filled_blank = int(t2.loc['  결측: 빈칸', '범위·전 연령'])
n_filled_dash = int(t2.loc["  결측: '-'", '범위·전 연령'])
if DENOM_MISSING == 'impute':
    clean = imputed
    assert clean.loc[in_scope(clean), '생활인구수'].notnull().all(), '보정 후에도 범위 안 결측이 남음(같은 구·연령대 3개 시간대가 모두 결측?)'
print(f"DENOM_MISSING = '{DENOM_MISSING}' 적용 | 범위 안 결측 남은 건수:", int((clean['생활인구수'].isnull() & in_scope(clean)).sum()),
      '| 범위 밖 결측(손대지 않음):', int((clean['생활인구수'].isnull() & ~in_scope(clean)).sum()))

_decision = '팀 확정(2026-10-09): 보정함' if DENOM_MISSING == 'impute' else '팀 확정: 보정 안 함'
_effect = (f'절대 순위는 보정 전후 동일(10대는 보정 대상 아님). 비중은 순위 변동 {rank_changed}/{N_GU}개 자치구, 상위5 구성 동일={top5_same}')

log.append(dict(처리='결측(빈칸): 범위 안 분모는 같은 구·연령대 평균으로 보정, 범위 밖은 그대로', 건수=flow('  결측: 빈칸'),
    근거='빈칸은 20·40·60대이상에만 있어 10대(분자)는 해당 없음. 그러나 비중의 분모(전 연령대)에 들어가므로 비워 두면 그 구의 분모가 작아져 비중이 부풀려짐',
    영향=f'{_decision} — 범위 안 빈칸 {n_filled_blank}건 보정. {_effect}. 범위 밖 빈칸은 영향이 없어 손대지 않음'))

log.append(dict(처리="결측('-'): 범위 안 10대는 해당 없음, 범위 안 분모는 같은 구·연령대 평균으로 보정, 범위 밖은 그대로", 건수=flow("  결측: '-'"),
    근거="10대 '-'는 범위(15~17시) 안에 0건이라 절대(합계)는 보정할 것이 없음(범위 안에 생기면 assert로 멈춤). 30·50대 '-'는 분모에 들어가므로 빈칸과 같은 기준으로 처리",
    영향=f"{_decision} — 범위 안 '-' {n_filled_dash}건 보정(30대·50대). 범위 밖 10대 '-'는 영향이 없어 손대지 않음"))

## E. 손대지 않는 것 — 확인만 하고 이유를 남긴다

In [ ]:
# ⑦ 동대문구 새벽 고값 (정확성) — 범위 밖이면 손대지 않는다
high = clean['생활인구수'] > 100000
print(f"10만 초과 값 {int(high.sum())}건 | 자치구 {clean.loc[high, '자치구'].unique().tolist()} | 시간대 {sorted(clean.loc[high, '시간대'].unique().tolist())}시")
n_high_scope = int((high & in_scope(clean)).sum())
print('범위(15~17시) 안 고값:', n_high_scope, '건')
assert n_high_scope == 0, '범위 안에 고값 발생 — 단위 오류(×1000) 처리 여부 결정 필요'

log.append(dict(처리='동대문구 새벽 고값(단위 오류 추정) — 수정하지 않음', 건수=flow('고값(>10만)'),
    근거=f"{sorted(clean.loc[high, '시간대'].unique().tolist())}시에만 발생, 인접 시간대의 수백 배(÷1000 하면 근접)로 단위 오류 추정",
    영향='손대지 않음 — 미션 범위(15~17시) 밖이라 절대·비중 어느 결과에도 영향 없음(비중의 분모도 15~17시 행만 사용). 수정하면 오히려 근거 없는 변경이 됨'))

# ⑧ 기준일자
dates = clean['기준일자'].unique().tolist()
print('기준일자 고유값:', dates)
assert len(dates) == 1

log.append(dict(처리='기준일자 형식·범위 확인 — 수정하지 않음', 건수='0건', 근거=f'전체 행이 {dates[0]} 단일 날짜, 형식 이상 없음',
    영향='손대지 않음 — 적시성 문제 없음. 단 하루치라는 점은 한계로 별도 명시'))

## F. 정제 전·후 점검 — 필터 전후 행 수 (0행이 나오는 무음 실패를 잡는 유일한 방법)

In [ ]:
m_after = mission(clean)
print(f"정제 전 전체 행 수               : {len(df)}")
print(f"정제 후 전체 행 수               : {len(clean)}")
print(f"미션 범위 필터 전 → 후 (정제 전) : {len(df)} → 전 연령대 {len(scope_all)}행 / 10대 {len(mission(df))}행")
print(f"미션 범위 필터 전 → 후 (정제 후) : {len(clean)} → 전 연령대 {int(in_scope(clean).sum())}행 / 10대 {len(m_after)}행")

assert len(m_after) > 0, '필터 결과가 0행 — 코드가 아니라 조건(시간대·연령대 값)을 의심'
assert len(m_after) == N_GU * N_HOURS
assert int(in_scope(clean).sum()) == N_GU * N_HOURS * N_AGES
assert clean.loc[in_scope(clean), '생활인구수'].notnull().all() and (clean.loc[in_scope(clean), '생활인구수'] >= 0).all()
assert m_after['자치구'].nunique() == N_GU
print('\n점검 통과: 범위 안은 자치구', N_GU, '× 시간대', N_HOURS, '× 연령대', N_AGES, '=', N_GU * N_HOURS * N_AGES, '행(전 연령대),',
      N_GU * N_HOURS, '행(10대), 결측·음수 없음')

pd.DataFrame(log, index=range(1, len(log) + 1))

## STEP⑤ 결론 — 절대·비중 교차 + 권고·근거·한계 (초안담당자: 정승엽 / 20점)

AI 프롬프트 예시:
- [절대] "15~18시·10대만 골라서 자치구별 합계를 구하고, 상위 5개를 표로 보여줘. 필터 전후 행 수도 함께 출력해줘."
- [비중] "같은 시간대 전 연령 대비 10대의 비중을 자치구별로 계산해 상위 5개를 보여줘. 절대 순위와 겹치는 구도 알려줘."


In [ ]:
# 절대 — 15~18시(=15·16·17시) · 10대 자치구별 합계
mission_df = mission(clean)
print('필터 전:', len(clean), '행 → 필터 후:', len(mission_df), '행')
absolute_rank = mission_df.groupby('자치구')['생활인구수'].sum().sort_values(ascending=False)
print('\n[절대] 전체 25개 자치구 순위:')
print(absolute_rank)
absolute_top5 = absolute_rank.head(5)
print('\n[절대] 상위 5:')
absolute_top5


In [ ]:
# 비중 — 같은 시간대(15·16·17시) 전 연령대 대비 10대 비중
same_time = clean[in_scope(clean)]
total_by_gu = same_time.groupby('자치구')['생활인구수'].sum()
teen_by_gu = mission_df.groupby('자치구')['생활인구수'].sum()
share_rank = (teen_by_gu / total_by_gu * 100).sort_values(ascending=False)
print('[비중] 전체 25개 자치구 순위(%):')
print(share_rank.round(2))
share_top5 = share_rank.head(5)
print('\n[비중] 상위 5:')
share_top5


In [ ]:
# 절대·비중 교차 — 두 기준 모두 상위인 자치구는 어디인가 + 가설(STEP①) 검증
print('절대 상위5:', list(absolute_top5.index))
print('비중 상위5:', list(share_top5.index))
cross_top5 = sorted(set(absolute_top5.index) & set(share_top5.index))
print('둘 다 상위5에 드는 자치구:', cross_top5)

# 상위5 밖으로 범위를 넓혀 추가 후보가 있는지 참고 확인
med_a, med_s = absolute_rank.median(), share_rank.median()
cross_pool = sorted(set(absolute_rank[absolute_rank >= med_a].index) & set(share_rank[share_rank >= med_s].index))
print('\n[참고] 절대·비중 모두 중앙값 이상인 교차 후보 풀:', cross_pool)

# 가설 검증 — 절대 상위3에 학원가(강남·양천·노원)가 몇 개 포함됐는가
hypothesis_gu = ['강남구', '양천구', '노원구']
abs_top3 = list(absolute_rank.head(3).index)
hit = [g for g in hypothesis_gu if g in abs_top3]
miss_gu = [g for g in hypothesis_gu if g not in abs_top3]
print('\n[가설 검증] 절대 상위3:', abs_top3)
print('학원가 목록 중 상위3 포함:', hit, '| 미포함:', miss_gu)
for g in hypothesis_gu:
    r_abs = list(absolute_rank.index).index(g) + 1
    r_share = list(share_rank.index).index(g) + 1
    print(f'  {g} — 절대 {r_abs}위({absolute_rank[g]:.0f}명) · 비중 {r_share}위({share_rank[g]:.1f}%)')


### 결론 3요소

**가설 검증**: 절대 상위3(강남·송파·관악) 중 가설이 예상한 학원가(강남·양천·노원)는 **강남 1곳만 포함**됐다(양천 9위, 노원 5위) — "다수 포함될 것"이라는 예측은 틀렸다. 송파(2위)는 이미 "번화한 학원가"로 분류해 둔 곳이라 설명 가능하지만, 관악구(3위)는 학원가 목록에 전혀 없던 **반대결과**다. 반면 비중은 가설이 잘 맞았다 — 강남(19위, 15.44%)·송파(17위, 15.88%) 모두 비중은 낮고, 노원은 2위(18.80%)로 "번화한 학원가일수록 비중은 희석된다"는 메커니즘이 그대로 확인됐다. 다만 비중 1위는 가설에 없던 영등포구(18.86%)였다.

- **권고**: **노원구·영등포구**를 확정 후보로, 세 번째는 판단보류로 둔다.
  - 노원구 — 절대 5위(15,710명)·비중 2위(18.80%): 가설이 예측한 "학원가+주거" 패턴과 일치
  - 영등포구 — 절대 4위(20,223명)·비중 1위(18.86%): 숫자·비율 모두 최상위권인 가장 강한 교차 근거. 다만 학원가 통념과는 무관해 추가 확인이 필요한 반대결과
  - 판단보류 — 절대 규모를 중시하면 **강남구**(1위, 27,203명. 이미 학원 인프라가 최상위라는 강점), 비중(집중도)을 중시하면 **중랑구**(비중 3위, 18.55%). 어느 기준을 우선할지는 정책 목적(최대 다수 vs 효율)에 따라 추가로 정해야 한다.

- **근거**: 10대 절대 합계와 비중을 교차한 결과, 양쪽 상위5에 모두 든 자치구는 **노원구·영등포구** 둘뿐이다(절대 상위5: 강남·송파·관악·영등포·노원 / 비중 상위5: 영등포·노원·중랑·서대문·금천). 중앙값 이상으로 범위를 넓히면 구로·금천·성북·양천·종로·중랑까지 추가 후보 풀에 들지만, 상위5 교차만큼 강하지는 않다.

- **한계**:
  1. 이 데이터는 2026-04-15 **단 하루**·교육용으로 오류가 의도적으로 삽입된 가공 데이터로, 실제 서울시 경향과 다를 수 있다.
  2. 생활인구는 **거주인구가 아니다** — 15~17시 그 구에 있던 10대가 그 구에 사는 아이들인지, 통학으로 온 다른 구 아이들인지는 이 데이터만으로 구분할 수 없다.
  3. "15~18시"를 15~17시로 해석했고(18시 버킷 제외), 정제 과정(완전 중복 제거·분모 결측 보정)에 따라 비중 순위가 25개 구 중 21개에서 바뀌었다 — 정제 판단에 결과가 민감하므로, 이 순위를 고정된 사실이 아니라 "이번 정제 기준에서의 결과"로 봐야 한다.
